In [1]:
import pickle
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses
from transformers import AutoTokenizer
from tqdm import tqdm
import os

# ==========================================
# 1. SETUP & LOAD DATA
# ==========================================
print("Loading datasets and trick questions...")
lang = "de"
model_name = 'intfloat/multilingual-e5-large'
batch_size = 16 # Kept small so Google Colab doesn't crash (Out of Memory)
epochs = 3

# Load your exact datasets
collection_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "collection", token="<HF_TOKEN>")["collection"]
train_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", lang, token="<HF_TOKEN>")["train"]

# Load the trick questions we made in Step 1
with open(f"hard_negs_{lang}.pkl", "rb") as f:
    hard_negs = pickle.load(f)

# Build a fast dictionary to look up documents by their pubkey
print("Organizing the library...")
doc_dict = {}
for item in collection_data:
    # We combine the fields with the [SEP] token just like the AIRwaves team!
    text = f"Title: {item['title']} [SEP] Abstract: {item['abstract']} [SEP] Venue: {item['venue']} [SEP] Authors: {item['authors']}"
    # Make it lowercase to keep it clean
    doc_dict[item['pubkey']] = " ".join(text.lower().split())

# ==========================================
# 2. THE SLICER (CHUNKING LOGIC)
# ==========================================
print("Loading the AI Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def slice_document(text):
    """Slices a long paper into 510-word chunks with a 50-word overlap."""
    enc = tokenizer(
        text,
        truncation=False,
        max_length=510,
        stride=50,
        return_overflowing_tokens=True
    )
    # Turn the numbers back into readable text chunks
    return [tokenizer.decode(ids, skip_special_tokens=True) for ids in enc["input_ids"]]

# ==========================================
# 3. BUILDING THE FLASHCARDS
# ==========================================
print("Creating training flashcards for the AI...")
training_flashcards = []

for item in tqdm(train_data):
    tweet_text = " ".join(item['text'].lower().split())
    correct_pubkey = item['pubkey']
    tweet_index = item['index']

    # 1. Get the Correct Document and slice it
    if correct_pubkey not in doc_dict:
        continue # Skip if the document is missing
    good_doc_text = doc_dict[correct_pubkey]
    good_chunks = slice_document(good_doc_text)

    # 2. Get the Trick Document and slice it
    trick_pubkey = hard_negs.get(tweet_index, [None])[0]
    trick_chunks = []
    if trick_pubkey and trick_pubkey in doc_dict:
        trick_doc_text = doc_dict[trick_pubkey]
        trick_chunks = slice_document(trick_doc_text)

    # 3. Create the combinations!
    # For every slice of the good doc, we pair it with a slice of the trick doc
    for good_slice in good_chunks:
        if trick_chunks:
            for trick_slice in trick_chunks:
                # Flashcard: [Tweet, Good Answer, Trick Answer]
                flashcard = InputExample(texts=[tweet_text, good_slice, trick_slice])
                training_flashcards.append(flashcard)
        else:
            # If we didn't find a trick answer, just show the good one
            flashcard = InputExample(texts=[tweet_text, good_slice])
            training_flashcards.append(flashcard)

# ==========================================
# 4. THE CLASSROOM (TRAINING LOOP)
# ==========================================
print("Setting up the AI model...")
model = SentenceTransformer(model_name)

# Memory saving tricks for Google Colab
try:
    tm = model._first_module().auto_model
    tm.gradient_checkpointing_enable()
    tm.config.use_cache = False
    print("Enabled memory-saving mode!")
except:
    pass

# Load the flashcards into a loader that shuffles them
train_dataloader = DataLoader(training_flashcards, shuffle=True, batch_size=batch_size)

# The Teacher: Multiple Negatives Ranking Loss
train_loss = losses.MultipleNegativesRankingLoss(model)

print(f"Starting training for {epochs} epochs! This will take some time...")
# Save the model in a new folder when it's done
save_path = f"./my_fine_tuned_e5_{lang}"
os.makedirs(save_path, exist_ok=True)

# 🚀 BLAST OFF
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=epochs,
    warmup_steps=int(len(train_dataloader) * epochs * 0.1), # Gently increase learning speed
    optimizer_params={'lr': 7e-6}, # Very small learning rate (safe)
    output_path=save_path,
    use_amp=True # Mixed precision (makes it run faster on GPUs)
)

print(f"🎉 Training Complete! Your custom AI is saved at: {save_path}")

Loading datasets and trick questions...


README.md: 0.00B [00:00, ?B/s]

collection_data.json:   0%|          | 0.00/36.5M [00:00<?, ?B/s]

Generating collection split:   0%|          | 0/10000 [00:00<?, ? examples/s]

de_train.json: 0.00B [00:00, ?B/s]

...

Enabled memory-saving mode!
Starting training for 3 epochs! This will take some time...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Training Complete! Your custom AI is saved at: ./my_fine_tuned_e5_de


In [2]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

print("1. Loading the Dev dataset...")
# Load the dev split for French (data the model has NEVER seen)
dev_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", lang, token="<HF_TOKEN>")["dev"]

print("2. Swapping out the generic AI for YOUR custom-trained AI...")
# Load your newly trained model from the folder you just saved it to
model_path = f"./my_fine_tuned_e5_{lang}"
model = SentenceTransformer(model_path)

# ==========================================
# PREPARE THE 10,000 DOCUMENTS
# ==========================================
print("3. Formatting the collection...")
doc_ids = []
doc_texts = []

for item in collection_data:
    # Must match the training format EXACTLY!
    text = f"Title: {item['title']} [SEP] Abstract: {item['abstract']} [SEP] Venue: {item['venue']} [SEP] Authors: {item['authors']}"
    doc_texts.append(" ".join(text.lower().split()))
    doc_ids.append(item['pubkey'])

print("4. Slicing and encoding 10k documents. This takes a few minutes...")
all_doc_embs = []

# torch.inference_mode() forces the GPU to save memory because we aren't training anymore
with torch.inference_mode():
    # We do this one document at a time to handle the dynamic chunk sizes
    for txt in tqdm(doc_texts):
        # 1. Slice the document (reusing the function from your previous cell!)
        chunks = slice_document(txt)

        # 2. Encode all chunks for this specific document
        chunk_embs = model.encode(chunks, batch_size=32, convert_to_numpy=True, show_progress_bar=False)

        # 3. The AIRwaves Trick: average the chunks, add the max spikes, divide by 2
        mean_emb = np.mean(chunk_embs, axis=0)
        max_emb = np.max(chunk_embs, axis=0)
        master_emb = (mean_emb + max_emb) / 2.0

        all_doc_embs.append(master_emb)

# Stack them into one giant matrix for fast math
doc_matrix = np.vstack(all_doc_embs)

# ==========================================
# PREPARE THE DEV QUERIES
# ==========================================
print("5. Encoding the Dev Queries...")
dev_queries = [" ".join(item['text'].lower().split()) for item in dev_data]
dev_labels = [item['pubkey'] for item in dev_data]

with torch.inference_mode():
    query_matrix = model.encode(dev_queries, batch_size=64, convert_to_numpy=True, show_progress_bar=True)

# ==========================================
# GRADE THE TEST (The Math)
# ==========================================
print("6. Calculating MRR@5 and Recall Scores...")

def compute_metrics(q_emb, doc_emb, gt_ids, doc_ids):
    sims = cosine_similarity(q_emb, doc_emb)
    ranks = []
    rr1, rr5, rr10 = [], [], []

    for i, row in enumerate(sims):
        # Sort scores descending
        order = np.argsort(-row)

        # Find where the correct document ID is in our sorted list
        try:
            gt_idx = doc_ids.index(gt_ids[i])
            rank = int(np.where(order == gt_idx)[0][0]) + 1
        except ValueError:
            rank = 10000 # Fallback if gold label is missing

        ranks.append(rank)
        rr1.append(1.0/rank if rank <= 1 else 0.0)
        rr5.append(1.0/rank if rank <= 5 else 0.0)
        rr10.append(1.0/rank if rank <= 10 else 0.0)

    ranks = np.array(ranks)
    return {
        "MRR@1": float(np.mean(rr1)),
        "MRR@5": float(np.mean(rr5)),
        "MRR@10": float(np.mean(rr10)),
        "Recall@5": float((ranks <= 5).mean()),
        "Recall@10": float((ranks <= 10).mean())
    }

final_scores = compute_metrics(query_matrix, doc_matrix, dev_labels, doc_ids)

print("\n🏆 === YOUR NEW BASELINE SCORES === 🏆")
for k, v in final_scores.items():
    print(f"{k:10s}: {v:.4f}")

1. Loading the Dev dataset...
2. Swapping out the generic AI for YOUR custom-trained AI...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

3. Formatting the collection...
4. Slicing and encoding 10k documents. This takes a few minutes...


100%|██████████| 10000/10000 [06:35<00:00, 25.28it/s]

5. Encoding the Dev Queries...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

6. Calculating MRR@5 and Recall Scores...

🏆 === YOUR NEW BASELINE SCORES === 🏆
MRR@1     : 0.4223
MRR@5     : 0.4921
MRR@10    : 0.4990
Recall@5  : 0.6140
Recall@10 : 0.6684
